# Prepare test-data score exports

In [1]:
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
from pandas.testing import assert_frame_equal

from manuscript_analysis import find_project_root
from manuscript_analysis.data import make_blocks
from manuscript_analysis.modeling import get_models

ROOT = find_project_root()
MODEL_DIR = ROOT / 'results' / 'matched_three_pipeline_run' / 'runs' / 'unsorted'
OUTPUT_DIR = ROOT / 'figures' / 'output'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

USED_FEATURES = [
    'Amplitude (uV)',
    'Trough to Peak (ms)',
    'Peak A (uV)',
    'Channel ISI (ms)',
]
BLOCK_SIZE = 500

assert MODEL_DIR.is_dir(), f'Model directory not found: {MODEL_DIR}'

In [2]:
models = get_models(MODEL_DIR)
list(models)

['seed0',
 'seed1',
 'seed2',
 'seed3',
 'seed4',
 'seed5',
 'seed6',
 'seed7',
 'seed8',
 'seed9',
 'seed10',
 'seed11',
 'seed12',
 'seed13',
 'seed14',
 'seed15',
 'seed16',
 'seed17',
 'seed18',
 'seed19',
 'seed20',
 'seed21',
 'seed22',
 'seed23',
 'seed24']

In [3]:
def confidence_weighted_session_score(block_scores, c=4, center=0.5):
    block_scores = np.asarray(block_scores, dtype=float)
    weights = np.exp(c * np.abs(block_scores - center))
    return float(np.average(block_scores, weights=weights))


def score_recording(recording_df):
    blocks = make_blocks(recording_df, features=USED_FEATURES, block_size=BLOCK_SIZE)
    scores = {}
    for model_id, (model, scaler, threshold) in models.items():
        scaled = ((np.clip(blocks, scaler['lo'], scaler['hi']) - scaler['mu']) / scaler['sd']).astype(np.float32)
        block_predictions = model.predict(scaled, batch_size=128, verbose=0).reshape(-1)
        scores[model_id] = (
            confidence_weighted_session_score(block_predictions),
            threshold,
        )
    return scores


def write_scores_with_parity_check(scores, output_path):
    if output_path.exists():
        previous = pd.read_csv(output_path)
        try:
            assert_frame_equal(previous, scores, check_exact=True)
        except AssertionError as error:
            raise AssertionError(
                f'Regenerated scores differ from the existing export: {output_path}'
            ) from error
        print(f'Parity verified against existing {output_path.name}')
    scores.to_csv(output_path, index=False)

## Drug-test scores

In [4]:
DRUG_DATA_DIR = ROOT / 'input' / 'drug_test'
DRUG_CONDITIONS = [
    ('AD-CO-DAY150-MKAD2-77-10NM-30DAYS', 'AD-CO-DAY150-MKAD2-77-10NM-30DAYS.pkl'),
    ('AD-CO-DAY150-MKAD2-77-100NM-30DAYS', 'AD-CO-DAY150-MKAD2-77-100NM-30DAYS.pkl'),
    ('OL-AD-CO-DAY150', 'OL-AD-CO-DAY150.pkl'),
]
MIN_SPIKES = 500
EXPECTED_DRUG_RECORDING_COUNTS = {
    'OL-AD-CO-DAY150': 6,
    'AD-CO-DAY150-MKAD2-77-10NM-30DAYS': 12,
    'AD-CO-DAY150-MKAD2-77-100NM-30DAYS': 13,
}

drug_rows = []
for condition, filename in DRUG_CONDITIONS:
    path = DRUG_DATA_DIR / filename
    frame = pd.read_pickle(path)
    recordings = frame.groupby('session', sort=False).size()
    selected = set(recordings.loc[recordings >= MIN_SPIKES].index)
    assert len(selected) == EXPECTED_DRUG_RECORDING_COUNTS[condition], (condition, len(selected))

    for recording, recording_df in frame.groupby('session', sort=False):
        if recording not in selected:
            continue
        for model_id, (score, threshold) in score_recording(recording_df).items():
            drug_rows.append({
                'condition': condition,
                'recording': recording,
                'model_id': model_id,
                'probability_ad': score,
                'optimal_threshold': threshold,
                'normalized_score': score - threshold,
            })

drug_scores = pd.DataFrame(drug_rows)
assert drug_scores.groupby(['condition', 'recording']).size().eq(25).all()
drug_output = OUTPUT_DIR / 'drug_test_model_recording_scores.csv'
write_scores_with_parity_check(drug_scores, drug_output)
print(f'Wrote {len(drug_scores):,} rows to {drug_output}')
drug_scores.groupby('condition').agg(recordings=('recording', 'nunique'), score_rows=('model_id', 'size'))

Wrote 775 rows to /Users/cuddysteadman/Desktop/DBC/manuscript-analysis/figures/output/drug_test_model_recording_scores.csv


,recordings,score_rows
condition,,
AD-CO-DAY150-MKAD2-77-100NM-30DAYS,13,325
AD-CO-DAY150-MKAD2-77-10NM-30DAYS,12,300
OL-AD-CO-DAY150,6,150


## Insertion-test scores

In [7]:
INSERTION_PATH = ROOT / 'input' / 'insertion' / 'insertion_dfs_Nov24.pkl'
INSERTION_CONDITIONS = [
    'AD-CO-DAY150',
    'AD-CO-DAY150-MKAD2-77-10NM-30DAYS',
    'AD-CO-DAY150-MKAD2-77-100NM-30DAYS',
]

with INSERTION_PATH.open('rb') as handle:
    insertion_data = pickle.load(handle)

missing = set(INSERTION_CONDITIONS).difference(insertion_data)
assert not missing, f'Missing insertion conditions: {sorted(missing)}'

insertion_rows = []
for class_name in INSERTION_CONDITIONS:
    for session, session_df in insertion_data[class_name].groupby('session', sort=False):
        for model_id, (score, threshold) in score_recording(session_df).items():
            insertion_rows.append({
                'session': session,
                'model_id': model_id,
                'optimal_threshold': threshold,
                'class_name': class_name,
                'class_label': 0 if class_name == 'CTRL-CO' else 1,
                'probability_ad': score,
            })

insertion_scores = pd.DataFrame(insertion_rows)
assert insertion_scores.groupby('session').size().eq(25).all()
insertion_output = OUTPUT_DIR / 'insertion_model_recording_scores.csv'
write_scores_with_parity_check(insertion_scores, insertion_output)
print(f'Wrote {len(insertion_scores):,} rows to {insertion_output}')
insertion_scores.groupby('class_name').agg(recordings=('session', 'nunique'), score_rows=('model_id', 'size'))

Wrote 275 rows to /Users/cuddysteadman/Desktop/DBC/manuscript-analysis/figures/output/insertion_model_recording_scores.csv


,recordings,score_rows
class_name,,
AD-CO-DAY150,2,50
AD-CO-DAY150-MKAD2-77-100NM-30DAYS,5,125
AD-CO-DAY150-MKAD2-77-10NM-30DAYS,4,100
